In [1]:
import pickle
import numpy as np

from qiskit.quantum_info import SparsePauliOp
from qiskit_qaoa.utils.string_utils import evaluate_sparse_pauli_samples
from qiskit_qaoa.utils.gfa_utils import gfa_file_to_graph

from qiskit_qaoa.hubo.graph_to_hubo_hamiltonian import graph_to_hubo_hamiltonian


from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import PauliEvolutionGate, CXGate
from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import (
    InverseCancellation,
    CommutativeCancellation
)

from qiskit_aer import AerSimulator
from qiskit_aer.backends.backendconfiguration import AerBackendConfiguration

from qopt_best_practices.transpilation.swap_cancellation_pass import SwapToFinalMapping
from qiskit_qaoa.utils.transpiler_passes import ExtendedSwapStrategy, FindCommutingPauliEvolutionsMulti
from qiskit_qaoa.utils.commuting_gate_router import CommutingGateRouter

In [2]:
# basepath='/lustre/scratch127/qpg/jc59/hubo/'
# filename='simulation.{}.optimisation.{}.extra{}.constraint{}.four{}.six{}.method{}.cvar{}.p{}.shots{}.init{}.d{}'.format(
#     'grid',
#     'test_N3_W4',
#     1,
#     1.0,
#     0.0,
#     1.0,
#     'Powell',
#     0.25,
#     4,
#     1000,
#     'random',
#     10
# )
# filepath = basepath + filename + '.pkl'
# with open(filepath, 'rb') as f:
#     data = pickle.load(f)
    
# history = data["history"]
# remapped_full_hamiltonian: SparsePauliOp = data["remapped_full_hamiltonian"]
# compiled_hamiltonian: SparsePauliOp = data["compiled_hamiltonian"]

In [3]:
# filepath = '/nfs/users/nfs_j/jc59/quantumwork/pangenome/data/trivial.gfa'
# graph, n, V, T = gfa_file_to_graph(filepath, [1,1,1])
# compiled_hamiltonian = graph_to_hubo_hamiltonian(graph, n, T, lamda=10, fraction_terms=1.0)
# len(compiled_hamiltonian)

In [4]:
# edge_map = {0: 5, 1: 10, 2: 8, 3: 4, 4: 0, 5: 2, 6: 9, 7: 7, 8: 11, 9: 1, 10: 3, 11: 6}
# inv_edge_map = {v: k for k, v in edge_map.items()}

In [5]:
# for sample in ['000000000000', '000010101000', '000011011011', '000111011111']:
#     path = [sample[i*3:(i+1)*3] for i in range(4)]
#     path = [(nodes[int(p[1]) + 2*int(p[2])], '+' if p[0] == '0' else '-') for p in path]
#     print(path)
#     print(evaluate_sparse_pauli_samples([sample], compiled_hamiltonian)[0])
#     # print(evaluate_sparse_pauli_samples([sample], remapped_full_hamiltonian.apply_layout([inv_edge_map[i] for i in range(12)]))[0])
#     remapped_sample_list = ['0'] * len(sample)
#     remapped_sample = ''.join([sample[11-inv_edge_map[11-j]] for j in range(12)])
#     # print(evaluate_sparse_pauli_samples([remapped_sample], remapped_full_hamiltonian)[0])
#     print(evaluate_sparse_pauli_samples([remapped_sample], compiled_hamiltonian.apply_layout([edge_map[i] for i in range(12)]))[0])

In [6]:
basepath = '/lustre/scratch127/qpg/jc59/hubo/'
filename = 'simulation.{}.compilation.{}.extra{}.constraint{}.four{}.six{}'.format(
    'grid',
    'trivial',
    1,
    1.0,
    0.0,
    1.0
)
results_file = basepath + filename + '.pkl'

with open(results_file, 'rb') as f:
    data = pickle.load(f)
    compiled_hamiltonian: SparsePauliOp = data['compiled_hamiltonian']
    full_hamiltonian: SparsePauliOp = data['full_hamiltonian']
    edge_map: dict[int, int] = data[0]

(0, 1) 1
(0, 1, 2) (0, 1) 1
(0, 1, 2, 4) (0, 1, 2) 1
(0, 1, 4) (0, 1, 2, 4) 1
(0, 1, 2, 4, 5) (0, 1, 4) 1
(1, 2, 4, 5) (0, 1, 2, 4, 5) 1
(1, 4) (1, 2, 4, 5) 1
(0, 1, 3, 4) (1, 4) 1
(1, 3, 4) (0, 1, 3, 4) 1
(1, 2, 3, 4) (1, 3, 4) 1

In [7]:
n = 3
T = 3
extended_swap_strat = ExtendedSwapStrategy.from_grid(n, T)
num_physical_qubits = extended_swap_strat._num_vertices

rng = np.random.default_rng(seed=1)
doubles = [set(x) for x in rng.choice(range(num_physical_qubits), (20, 2))] + [set(x) for x in rng.choice(range(num_physical_qubits), (10, 1))] + [tuple()]
quads = [set(x) for x in rng.choice(range(num_physical_qubits), (20, 4))] + [set(x) for x in rng.choice(range(num_physical_qubits), (20, 6))]
# doubles = [(0,1),(0,1,2),(0,1,2,4),(0,1,4),(0,1,2,4,5),(1,2,4,5),(1,4),(0,1,3,4),(1,3,4),(1,2,3,4)] 
# quads = []
coeffs = rng.choice(np.linspace(0.125, 1, 8), len(doubles)+len(quads))
def choice_to_pauli(choice: list[int] | tuple[int,...], size: int) -> str:
    pauli = ['I'] * size
    for x in choice:
        pauli[size - x - 1] = 'Z'
    return ''.join(pauli)

hamiltonian = SparsePauliOp([choice_to_pauli(c, num_physical_qubits) for c in doubles] + [choice_to_pauli(c, num_physical_qubits) for c in quads], coeffs=coeffs)
hamiltonian = hamiltonian.simplify()
hamiltonian = hamiltonian.sort(weight=True)

In [8]:
basis_gates=["sx", "x", "rz", "rzz", "cz", "id", "swap", "cx", "h"]

backend_options = dict(
    method='unitary',
    device='CPU',
    precision='single',
    basis_gates=basis_gates,
)


config = AerSimulator._DEFAULT_CONFIGURATION
config["n_qubits"] = num_physical_qubits
config["basis_gates"] = basis_gates
config = AerBackendConfiguration.from_dict(config)
backend = AerSimulator(configuration=config, coupling_map=extended_swap_strat._coupling_map, **backend_options)
backend.set_option("n_qubits", num_physical_qubits)
print(f'Num qubits in backend: {backend.configuration().to_dict()["n_qubits"]}')

Num qubits in backend: 9


(0, 1) Instruction(name='PauliEvolution', num_qubits=2, num_clbits=0, params=[np.float64(-2.8125)])
(0, 1, 4) Instruction(name='PauliEvolution', num_qubits=3, num_clbits=0, params=[np.float64(0.4375)])
(0, 4) 0
(0, 4) Instruction(name='PauliEvolution', num_qubits=2, num_clbits=0, params=[np.float64(-0.4375)])
(0, 1) 0
(0, 1, 2, 3, 4) Instruction(name='PauliEvolution', num_qubits=5, num_clbits=0, params=[np.float64(0.625)])
(0, 1, 2, 3) 0
(1, 2, 3) Instruction(name='PauliEvolution', num_qubits=3, num_clbits=0, params=[np.float64(0.625)])
(0, 4) 0

In [9]:
swap_depth = 0
pm = PassManager(
    [
        FindCommutingPauliEvolutionsMulti(), 
        CommutingGateRouter(
            extended_swap_strat,
            max_layers=swap_depth,
            perform_extra_swaps=True
        ),
        SwapToFinalMapping(),
        InverseCancellation(gates_to_cancel=[CXGate()]),
        CommutativeCancellation(basis_gates=["cx", "swap", "rz"]),
        InverseCancellation(gates_to_cancel=[CXGate()]),
    ]
)


cost_qc = QuantumCircuit(num_physical_qubits)
cost_qc.append(PauliEvolutionGate(hamiltonian), range(num_physical_qubits))
# cost_qc.append(PauliEvolutionGate(compiled_hamiltonian), [edge_map[i] for i in range(num_physical_qubits)])
# cost_qc.append(PauliEvolutionGate(compiled_hamiltonian), range(num_physical_qubits))


tcost_qc = pm.run(cost_qc)
tcost_qc.save_unitary()
cost_qc.save_unitary()

Max layers needed to apply swap decompose: 0
Applying phase: Instruction(name='PauliEvolution', num_qubits=0, num_clbits=0, params=[np.float64(0.625)])
Computing site: (0, 1) onto 1 with current info: {0: {0}, 1: {1}, 2: {2}, 3: {3}, 4: {4}, 5: {5}, 6: {6}, 7: {7}, 8: {8}}
Candidate: 0
Applied gates: [(0, 1)]
Fixed qubits: {0}
Computing site: (0, 1, 4, 7) onto 1 with current info: {0: {0}, 1: {0, 1}, 2: {2}, 3: {3}, 4: {4}, 5: {5}, 6: {6}, 7: {7}, 8: {8}}
Candidate: 4
Applied gates: [(7, 4), (4, 1)]
Fixed qubits: {4, 7}
Computing site: (1, 2) onto 1 with current info: {0: {0}, 1: {1}, 2: {2}, 3: {3}, 4: {4}, 5: {5}, 6: {6}, 7: {7}, 8: {8}}
Candidate: 2
Applied gates: [(2, 1)]
Fixed qubits: {2}
Computing site: (4, 5) onto 4 with current info: {0: {0}, 1: {1, 2}, 2: {2}, 3: {3}, 4: {4}, 5: {5}, 6: {6}, 7: {7}, 8: {8}}
Candidate: 5
Applied gates: [(5, 4)]
Fixed qubits: {5}
Computing site: (0, 3, 4, 5) onto 4 with current info: {0: {0}, 1: {1, 2}, 2: {2}, 3: {3}, 4: {4, 5}, 5: {5}, 6: {6},

In [10]:
cost_for_backend = transpile(cost_qc, optimization_level=0, backend=backend)
tcost_for_backend = transpile(tcost_qc, optimization_level=0, backend=backend)

In [11]:
cost_qc.decompose().draw(fold=-1)

global phase: 5.6582
     ┌──────────┐                                                                                                                                                                                                                                                                                                                                                                               ┌───┐┌─────────┐┌───┐┌───┐┌──────────┐┌───┐                    ┌───┐┌──────────┐┌───┐                    ┌───┐┌─────────┐┌───┐                    ┌───┐┌──────────┐   ┌───┐                                 ┌───┐┌──────────┐┌───┐                                                                                                                                                       ┌───┐┌──────────┐┌───┐                                               ┌───┐┌───────┐┌───┐                                                           ┌───┐┌───────┐┌───┐                                                              ┌───┐┌──────────┐┌───┐                    ┌───┐┌─────────┐┌───┐                    ┌───┐┌──────────┐┌───┐                              ┌───┐┌───────┐┌───┐                              ┌───┐┌─────────┐┌───┐                                                                                                                                                                                         ┌───┐┌─────────┐┌───┐                              ┌───┐┌───────┐┌───┐                                                                                              ┌───┐┌───────┐┌───┐                                        ┌───┐┌───────┐┌───┐                     unitary 
q_0: ┤ Rz(3.75) ├─■─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤ X ├┤ Rz(3.5) ├┤ X ├┤ X ├┤ Rz(0.25) ├┤ X ├────────────────────┤ X ├┤ Rz(0.25) ├┤ X ├────────────────────┤ X ├┤ Rz(1.5) ├┤ X ├────────────────────┤ X ├┤ Rz(1.75) ├───┤ X ├─────────────────────────────────┤ X ├┤ Rz(1.75) ├┤ X ├───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤ X ├┤ Rz(0.75) ├┤ X ├───────────────────────────────────────────────┤ X ├┤ Rz(2) ├┤ X ├───────────────────────────────────────────────────────────┤ X ├┤ Rz(2) ├┤ X ├──────────────────────────────────────────────────────────────┤ X ├┤ Rz(0.75) ├┤ X ├────────────────────┤ X ├┤ Rz(0.5) ├┤ X ├────────────────────┤ X ├┤ Rz(0.25) ├┤ X ├──────────────────────────────┤ X ├┤ Rz(2) ├┤ X ├──────────────────────────────┤ X ├┤ Rz(1.5) ├┤ X ├─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤ X ├┤ Rz(1.5) ├┤ X ├──────────────────────────────┤ X ├┤ Rz(1) ├┤ X ├──────────────────────────────────────────────────────────────────────────────────────────────┤ X ├┤ Rz(2) ├┤ X ├────────────────────────────────────────┤ X ├┤ Rz(1) ├┤ X ├────────────────────────░────
     └┬───────┬─┘ │ZZ(0.5)                                                                                                                                                                                             ┌───┐┌───────┐┌───┐                                         ┌───┐┌─────────┐┌───┐          ┌───┐┌───────┐┌───┐┌───┐┌──────────┐┌───┐                      ┌───┐          └─┬─┘└─────────┘└─┬─┘└─┬─┘└──────────┘└─┬─┘┌───┐               └─┬─┘└──────────┘└─┬─┘                    └─┬─┘└─────────┘└─┬─┘               ┌───┐└─┬─┘└──────────┘   └─┬─┘   ┌───┐                    ┌───┐└─┬─┘└──────────┘└─┬─┘┌───┐               ┌───┐┌──────────┐┌───┐                    ┌───┐

In [12]:
tcost_qc.draw(fold=-1, idle_wires=False)

global phase: 5.6582
         ┌──────────┐                                            {0, 1, 4, 7}                       {1, 2}                                            {0, 3, 4, 5}                        {6, 7}                                                                                              {1, 3, 4, 5, 7}                                              {8, 4, 7}                                                       {0, 2, 3, 4, 5}                                                                 {1, 3, 4, 6, 7}                                                                {0, 3, 4, 5, 6}                ┌───┐┌──────────┐┌───┐                {0, 1, 3, 4, 5}                                                      {2, 3, 4, 5, 6}                          ┌───┐┌───────┐┌───┐                          {0, 3, 4, 7, 8}  ░                         ┌───┐┌──────────┐┌───┐                                                                                                                                                                                                                                                                      ┌───┐┌──────────┐┌───┐                                                                                                                                                   ┌───┐                                                                                                                                                                                                                                                                                                                   ┌───┐                                         ┌───┐                                                                                                                                                          ┌───┐                     unitary 
q_0 -> 0 ┤ Rz(3.75) ├──■─────────────────────────────────────■────────░───────────────────────────────░──────■────────────────────────────────────■────────░────────────────────────────────░────────────────────────────────────────────────────────────────────────────────────────────────────────░─────────────────────────────────────────────────────────░───────■──────────────────────────────────────────■───────────────░───────────────────────────────────────────────────────────────────────────────░────────────────────────────────────■───────────────■─────────────────────────░───────────────────────┤ X ├┤ Rz(0.25) ├┤ X ├───────────────────────░────────────────────────────────────────────────────────────────────░─────────────────────────────────┤ X ├┤ Rz(1) ├┤ X ├─────────────────────────────────░─────────░─────────────────────────┤ X ├┤ Rz(0.75) ├┤ X ├─────────────────────────────────────────────────────────────────────────────────────────X─────────────────────────────────────────────────X────────────X───────────────────────────────X─────────────────────────────────────────────────────────────────────────────┤ X ├┤ Rz(0.75) ├┤ X ├───────────────────────────────────────────────────────────────X──────────────────────────X───X────────────────────────────────────────────────────┤ X ├──X───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────■─────────────────────────■───────────────────────────────────────────────────────────────────────────────────────────────■────■─────────────────────────────────■───X───────────┤ X ├──■───────────────────────────────────■──┤ X ├──X──────────────────────────────────────────────■────X──────────────────────────────────────────────■──────────────────────────────────────────────■─────┤ X ├─────■──────────────────░────
         └┬───────┬─┘┌─┴─┐┌─────────┐┌───┐┌──────────┐┌───┐┌─┴─┐      ░       ┌───┐┌─────────┐┌───┐   ░      │                                    │        ░                                ░                                                                                                        ░ 

In [13]:
res_cost = backend.run(cost_for_backend).result()

In [14]:
data = res_cost.results[0].data
unitary = np.asarray(data.unitary)

In [15]:
res_tcost = backend.run(tcost_for_backend).result()

In [16]:
tdata = res_tcost.results[0].data
tunitary = np.asarray(tdata.unitary)

In [17]:
scores = {i: np.exp(-1j*i) for i in np.linspace(-100,100, 1601)}
def get_score(x: complex):
    for i in np.linspace(-100,100, 1601):
        if np.abs(scores[i]- x) < 1e-6:
            return i 
    return x

In [18]:
# unitary = unitary/ unitary[np.nonzero(unitary)][0]
weights = []
for entry in np.transpose(np.nonzero(unitary)):
    weights.append((np.binary_repr(entry[1], num_physical_qubits), get_score(unitary[tuple(entry)])))
weights = sorted(weights, key=lambda e: e[0])
for w in weights[:10]:
    print(w[0], w[1])
    # print(evaluate_sparse_pauli_samples([w[0]], compiled_hamiltonian)[0])
    
# print()
# 000010101000 -> 8+32+128 = 168
# for w in weights[167:172]:
#     print(w)

000000000 42.125
000000001 13.625
000000010 11.625
000000011 2.125
000000100 14.625
000000101 7.625
000000110 3.125
000000111 -4.875
000001000 19.125
000001001 12.625


In [19]:
mod_tunitary = tunitary
# mod_tunitary = tunitary * np.exp(-1j * np.float64(-0.7535131086748066))
# mod_tunitary = (tunitary / tunitary[np.nonzero(tunitary)][2]) # * unitary[np.nonzero(unitary)][0]
tweights = []
for entry in np.transpose(np.nonzero(np.abs(mod_tunitary) > 1e-6)):
    tweights.append((np.binary_repr(entry[1], num_physical_qubits), get_score(mod_tunitary[tuple(entry)])))
tweights = sorted(tweights, key=lambda e: e[0])
for w in tweights[:10]:
    print(w[0], w[1])

000000000 42.125
000000001 13.625
000000010 10.625
000000011 3.125
000000100 14.625
000000101 7.625
000000110 2.125
000000111 -3.875
000001000 20.125
000001001 11.625


Some kind of non-integer rotation effect is taking place.
Certain indices stay "in sync" e.g. when we force the first one to 0, several others return a score which actually corresponds to the true score when shifted by -36 (the score of 00...0)

But others are not in the right phase.

In [20]:
np.nonzero(np.abs(np.array([w[1] for w in weights]) - np.array([w[1] for w in tweights])) >= 1e-6)

(array([  2,   3,   6,   7,   8,   9,  12,  13,  18,  19,  22,  23,  24,
         25,  28,  29,  32,  33,  36,  37,  42,  43,  46,  47,  48,  49,
         52,  53,  58,  59,  62,  63,  64,  65,  68,  69,  74,  75,  78,
         79,  80,  81,  84,  85,  90,  91,  94,  95,  98,  99, 102, 103,
        104, 105, 108, 109, 114, 115, 118, 119, 120, 121, 124, 125, 130,
        131, 134, 135, 136, 137, 140, 141, 146, 147, 150, 151, 152, 153,
        156, 157, 160, 161, 164, 165, 170, 171, 174, 175, 176, 177, 180,
        181, 186, 187, 190, 191, 192, 193, 196, 197, 202, 203, 206, 207,
        208, 209, 212, 213, 218, 219, 222, 223, 226, 227, 230, 231, 232,
        233, 236, 237, 242, 243, 246, 247, 248, 249, 252, 253, 258, 259,
        262, 263, 264, 265, 268, 269, 274, 275, 278, 279, 280, 281, 284,
        285, 288, 289, 292, 293, 298, 299, 302, 303, 304, 305, 308, 309,
        314, 315, 318, 319, 320, 321, 324, 325, 330, 331, 334, 335, 336,
        337, 340, 341, 346, 347, 350, 351, 354, 355

In [21]:
ratio = unitary[0,0] / tunitary[0,0]
np.arctan(np.imag(ratio) / np.real(ratio))

np.float64(-4.0330999009834835e-08)

In [22]:
ratio = weights[0][1] / tweights[0][1] 
np.arctan(np.imag(ratio) / np.real(ratio))

np.float64(0.0)

For random circuit:

When skipping phase
np.arctan(np.imag(ratio) / np.real(ratio)) = np.float64(-0.7535130931655948)

When applying phase
np.arctan(np.imag(ratio) / np.real(ratio)) = np.float64(0.7535130931655948)

When applying half-phase
np.arctan(np.imag(ratio) / np.real(ratio))  = 0 